# basic 

In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
import polars as pl
import numpy as np
import pickle
from tqdm import tqdm

import src.configs as configs
from src.tokenizer import GraphTokenizer


# load data and build tokenizer

In [ ]:
with open(configs.ProcessedGraph().candidate_is_a_reachable_dict, "rb") as f:
    candidate_reachable_child_map = pickle.load(f)

df_concept_hug_w_depth = pl.read_parquet(configs.ProcessedGraph.concept_w_depth)
mapped_candidate_rel_dist_prop = (pl.read_parquet(configs.ProcessedGraph().mapped_candidate_rel_dist_prop)
                                  
                                  # remove all relations in subgraph where candidate and mapped are distant
                                  .filter(pl.col("distance") <= configs.TokenizerParam().max_dist_candidate)
                                  )

tokenizer = GraphTokenizer(df_concept_hug_w_depth,
                        mapped_candidate_rel_dist_prop,
                        candidate_reachable_child_map,
                        configs.TokenizerParam().max_dist_candidate
                        )

Ks = np.arange(500, 12000, 500)

# load diff candidate set

In [ ]:
# baselines
baseline_candidates = {
"b_highest_deg_list" : pl.read_parquet(f"{configs.Baselines().path}highest_degree.parquet")["dst.id"].to_list(),
"b_highest_deg_dist_1_list" : pl.read_parquet(f"{configs.Baselines().path}highest_degree_dist_1.parquet")["dst.id"].to_list(),
"b_most_children_list" : pl.read_parquet(f"{configs.Baselines().path}most_children.parquet")["candidate"].to_list(),
"b_random_k_all" : pl.read_parquet(f"{configs.Baselines().path}k_random_all_samples.parquet"),
}
# graph red 
graph_red_candidates = {
    "gr_sum_inv": pl.read_parquet(f"{configs.IterativeGraphRed().path} sum_inv.parquet")["candidate"].to_list(),
    "gr_sum_inv_exp": pl.read_parquet(f"{configs.IterativeGraphRed().path} sum_inv_exp.parquet")["candidate"].to_list(),
    "gr_tempered_sum_inv": pl.read_parquet(f"{configs.IterativeGraphRed().path} tempered_sum_inv.parquet")["candidate"].to_list(),
    "gr_tempered_sum_inv_exp": pl.read_parquet(f"{configs.IterativeGraphRed().path} tempered_sum_inv_exp.parquet")["candidate"].to_list(),

}

# marginal gain
margin_gain_candidates = {
    "mg_inv" : pl.read_parquet(f"{configs.IterativeMarginalGain().path}_inv.parquet")["candidate_id"].to_list(),
    "mg_inv_exp" : pl.read_parquet(f"{configs.IterativeMarginalGain().path}_inv_exp.parquet")["candidate_id"].to_list(),
    
}

all_candidates = [baseline_candidates, graph_red_candidates, margin_gain_candidates]

In [26]:
results = []

# methods that produce one ordered list of candidates (rank/priority order);
# performance per k = evaluate the top-k of that ordered list
ordered_candidate_sets = {
    **{name: ids for name, ids in baseline_candidates.items() if name != "b_random_k_all"},
    **graph_red_candidates,
    **margin_gain_candidates,
}

for name, ordered_ids in ordered_candidate_sets.items():
    print(name)
    for k in tqdm(Ks):
        scores, _, _ = tokenizer.evaluate_components_and_tokenize(ordered_ids[:k])
        results.append({"method": name, "k": k, **scores})


b_highest_deg_list


100%|██████████| 23/23 [01:12<00:00,  3.16s/it]


b_highest_deg_dist_1_list


100%|██████████| 23/23 [00:53<00:00,  2.31s/it]


b_most_children_list


...

100%|██████████| 23/23 [00:49<00:00,  2.16s/it]


mg_inv


100%|██████████| 23/23 [00:55<00:00,  2.43s/it]


mg_inv_exp


100%|██████████| 23/23 [00:52<00:00,  2.29s/it]


In [ ]:
# random-k baseline: average performance over all simulations of the same k
random_k_df = baseline_candidates["b_random_k_all"]

for k in Ks:
    iter_scores = []
    for s in random_k_df.filter(pl.col("k") == k)["iter"].unique().sort():
        candidate_ids = random_k_df.filter((pl.col("k") == k) & (pl.col("iter") == s))["candidate_id"].to_list()
        scores, _, _ = tokenizer.evaluate_components_and_tokenize(candidate_ids)
        iter_scores.append(scores)
    mean_scores = pl.DataFrame(iter_scores).mean().to_dicts()[0]
    results.append({"method": "b_random_k", "k": k, **mean_scores})

df_performance = pl.DataFrame(results)
df_performance